This script creates a complete, runnable Jupyter notebook that demonstrates; how to run media optimization & simulation scenarios for marketing allocation. It also saves a lightweight CSV input template you can replace with your own data.

In [22]:
# -----------------------------
# Create a synthetic CSV input
# -----------------------------
channels = ["paid_search","display","affiliate","social","email"]
weeks = list(range(1, 14))  # one quarter ~13 weeks

In [23]:
import pandas as pd

rows = []
for w in weeks:
    for ch in channels:
        base_impr = {
            "paid_search": 8_000_000,
            "display": 12_000_000,
            "affiliate": 5_000_000,
            "social": 9_000_000,
            "email": 2_500_000,
        }[ch]
        cpi = {
            "paid_search": 0.012,
            "display": 0.004,
            "affiliate": 0.006,
            "social": 0.003,
            "email": 0.0015,
        }[ch]
        freq_cap = {
            "paid_search": 4,
            "display": 5,
            "affiliate": 4,
            "social": 2,
            "email": 6,
        }[ch]
        # CTR and CVR assumptions (toy; will be used inside notebook)
        ctr = {
            "paid_search": 0.035,
            "display": 0.004,
            "affiliate": 0.018,
            "social": 0.008,
            "email": 0.05,
        }[ch]
        cvr = {
            "paid_search": 0.055,
            "display": 0.012,
            "affiliate": 0.030,
            "social": 0.016,
            "email": 0.040,
        }[ch]
        rows.append({
            "week": w,
            "channel": ch,
            "base_impressions": int(base_impr * (0.85 + 0.3 * (w % 3 == 0))), # mild seasonality
            "cpi": cpi,
            "freq_cap_per_user": freq_cap,
            "ctr": ctr,
            "cvr": cvr,
            "avg_order_value": 95.0,             # AOV constant for simplicity
            "existing_users": 250_000           # user pool per week (toy)
        })

df = pd.DataFrame(rows)

df.to_csv(f"input_file.csv", index=False)


In [24]:
df.head()

,week,channel,base_impressions,cpi,freq_cap_per_user,ctr,cvr,avg_order_value,existing_users
0,1,paid_search,6800000,0.0120,4,0.035,0.055,95.0,250000
1,1,display,10200000,0.0040,5,0.004,0.012,95.0,250000
2,1,affiliate,4250000,0.0060,4,0.018,0.030,95.0,250000
3,1,social,7650000,0.0030,2,0.008,0.016,95.0,250000
4,1,email,2125000,0.0015,6,0.050,0.040,95.0,250000


In [8]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

# Settings for reproducibility
np.random.seed(42)

# Load inputs (replace with your file)
csv_path = "input_file.csv"
try:
    # Try to load your path first
    df = pd.read_csv(csv_path)
except Exception:
    # Fall back to bundled sample
    import os, glob
    candidates = sorted(glob.glob('/data/sample_media_inputs_*.csv'))
    df = pd.read_csv(candidates[-1])
    
df.head()

,week,channel,base_impressions,cpi,freq_cap_per_user,ctr,cvr,avg_order_value,existing_users
0,1,paid_search,6800000,0.0120,4,0.035,0.055,95.0,250000
1,1,display,10200000,0.0040,5,0.004,0.012,95.0,250000
2,1,affiliate,4250000,0.0060,4,0.018,0.030,95.0,250000
3,1,social,7650000,0.0030,2,0.008,0.016,95.0,250000
4,1,email,2125000,0.0015,6,0.050,0.040,95.0,250000


# Media Optimization & Simulation Research Project

**Objective:** Provide a practical, reproducible framework to **run optimization and simulation scenarios** that inform:
- Quarterly **marketing investment & allocation** recommendations
- **Media plan** inputs, **financial forecasting**, and **efficiency gains**
- Insights on **Cost to Acquire (CAC)**, **value of digital engagement**, and **cross-channel impact**

> Replace the sample CSV with your data to run this end-to-end. The notebook includes: data ingestion, adstock & response curves, greedy optimization with budget & frequency caps, quarterly scenarios, Monte Carlo sensitivity, and reporting.

In [19]:
import pandas as pd

# Create a sample media DataFrame
data = {
    "channel": ["TV", "Social", "Search", "Display", "Email", "Affiliate"],
    "cost": [300000, 200000, 150000, 100000, 50000, 20000],
    "conversions": [1200, 1500, 2000, 800, 400, 300],
    "revenue": [900000, 700000, 600000, 250000, 100000, 80000]
}

media_df = pd.DataFrame(data)

# Save to CSV
file_path = "sample_media_inputs_20250822_040242.csv"
media_df.to_csv(file_path, index=False)

file_path

'sample_media_inputs_20250822_040242.csv'

## 2) Helper functions: adstock, saturation (diminishing returns), and KPIs

In [17]:
import pandas as pd

def greedy_optimizer(panel, budget, objective="roi", freq_cap=None):
    """
    Simple greedy optimizer for marketing allocation.
    - panel: DataFrame with channel-level cost, conversions, revenue, ROI info
    - budget: total spend budget to allocate
    - objective: "roi" or "lift" (revenue lift)
    - freq_cap: optional max number of increments per channel
    
    Returns:
        alloc_detail: DataFrame of allocations
        summary: Aggregated totals
    """
    allocs = []
    remaining_budget = budget
    
    # copy to avoid modifying original
    df = panel.copy()
    
    # sort by ROI or revenue per cost
    if objective == "roi":
        df["priority"] = df["roi"]
    elif objective == "lift":
        df["priority"] = df["revenue"] / df["cost"]
    else:
        raise ValueError("objective must be 'roi' or 'lift'")
    
    df = df.sort_values("priority", ascending=False).reset_index(drop=True)
    
    for idx, row in df.iterrows():
        if remaining_budget <= 0:
            break
        
        alloc = min(row["cost"], remaining_budget)
        
        # apply frequency cap (if given)
        if freq_cap is not None:
            alloc = min(alloc, row["cost"] * freq_cap)
        
        conversions = row["conversions"] * (alloc / row["cost"])
        revenue = row["revenue"] * (alloc / row["cost"])
        
        allocs.append({
            "channel": row["channel"],
            "allocated_cost": alloc,
            "conversions": conversions,
            "revenue": revenue,
            "roi": revenue / alloc if alloc > 0 else 0
        })
        
        remaining_budget -= alloc
    
    alloc_detail = pd.DataFrame(allocs)
    
    summary = alloc_detail.agg({
        "allocated_cost": "sum",
        "conversions": "sum",
        "revenue": "sum"
    }).to_dict()
    summary["roi"] = summary["revenue"] / summary["allocated_cost"]
    
    return alloc_detail, summary

In [20]:
panel = pd.read_csv("sample_media_inputs_20250822_040242.csv")

alloc_detail, summary = greedy_optimizer(
    panel, 
    budget=1_000_000,   # planning budget
    objective="roi",    # or "lift"
    freq_cap=5          # optional
)

KeyError: 'roi'

In [ ]:
def apply_allocation_and_score(panel: pd.DataFrame, alloc_detail: pd.DataFrame) -> pd.DataFrame:
    df2 = alloc_detail.copy()
    df2['new_impressions'] = df2['base_impressions'] + df2['extra_impressions']
    
    outs = []
    for ch in channels:
        sub = df2[df2.channel==ch].copy().sort_values('week')
        impr = sub['new_impressions'].to_numpy()
        adx = adstock(impr, decay=0.5)
        sat = hill_saturation(adx, alpha=1.0, half_saturation=np.percentile(adx, 75))
        k = kpis_from_impressions(impr, sub['ctr'].iloc[0], sub['cvr'].iloc[0], sub['avg_order_value'].iloc[0])
        cost = cost_from_impressions(impr, sub['cpi'].iloc[0])
        d = pd.DataFrame({
            'week': sub['week'].values,
            'channel': ch,
            'impressions': impr,
            'conversions': k['conversions'],
            'revenue': k['revenue'],
            'cost': cost
        })
        outs.append(d)
    scored = pd.concat(outs, ignore_index=True)
    return scored


In [15]:
scored = apply_allocation_and_score(panel, alloc_detail)
baseline_summary = baseline.groupby('channel', as_index=False).agg(cost=('cost','sum'), conv=('conversions','sum'), rev=('revenue','sum'))
scenario_summary = scored.groupby('channel', as_index=False).agg(cost=('cost','sum'), conv=('conversions','sum'), rev=('revenue','sum'))

summary = baseline_summary.merge(scenario_summary, on='channel', suffixes=('_base','_new'))
summary['delta_cost'] = summary['cost_new'] - summary['cost_base']
summary['delta_conv'] = summary['conv_new'] - summary['conv_base']
summary['delta_rev'] = summary['rev_new'] - summary['rev_base']
summary['marginal_cac'] = summary['delta_cost'] / summary['delta_conv'].replace(0, np.nan)
summary['marginal_roas'] = summary['delta_rev'] / summary['delta_cost'].replace(0, np.nan)
summary

NameError: name 'alloc_detail' is not defined

## 6) Quarterly scenarios & forecasting

In [13]:
def run_quarterly_scenarios(panel, budgets):
    results = {}
    for name, b in budgets.items():
        det, _ = greedy_optimize(panel, b, {ch:30 for ch in channels}, objective='roi')
        sc = apply_allocation_and_score(panel, det)
        s = sc.groupby('channel', as_index=False).agg(cost=('cost','sum'), conv=('conversions','sum'), rev=('revenue','sum'))
        results[name] = s
    return results

budgets = {
    "flat": 0,
    "base+250k": 250_000,
    "base+500k": 500_000,
    "base+1M": 1_000_000
}

scenarios = run_quarterly_scenarios(panel, budgets)

# Compare total KPIs
def kpi_totals(df):
    return pd.Series({"cost": df['cost'].sum(), "conv": df['conv'].sum(), "rev": df['rev'].sum(), "roas": df['rev'].sum()/df['cost'].sum()})

totals = pd.DataFrame({name: kpi_totals(scenarios[name]) for name in scenarios}).T
totals

NameError: name 'greedy_optimize' is not defined

## 7) Sensitivity analysis (Monte Carlo on CPI & elasticities)

In [ ]:
def monte_carlo(panel, budget, n=200, cpi_sd=0.15, ctr_sd=0.20, cvr_sd=0.20):
    roas_list = []
    for _ in range(n):
        noisy = panel.copy()
        for ch in channels:
            mask = noisy.channel==ch
            noisy.loc[mask,'cpi'] *= np.random.lognormal(mean=0, sigma=cpi_sd)
            noisy.loc[mask,'ctr'] *= np.random.lognormal(mean=0, sigma=ctr_sd)
            noisy.loc[mask,'cvr'] *= np.random.lognormal(mean=0, sigma=cvr_sd)
        det, _ = greedy_optimize(noisy, budget, {ch:30 for ch in channels}, objective='roi')
        sc = apply_allocation_and_score(noisy, det)
        roas_list.append(sc['revenue'].sum()/sc['cost'].sum())
    return np.array(roas_list)

mc = monte_carlo(panel, 500_000, n=200)

import matplotlib.pyplot as plt
plt.figure()
plt.hist(mc, bins=25)
plt.title("Monte Carlo ROAS distribution (budget=500k)")
plt.xlabel("ROAS")
plt.ylabel("Count")
plt.show()

mc.mean(), mc.std()

## 8) Cross-channel impact via halo matrix

In [ ]:
H = halo_spillover_matrix(channels)

def apply_halo(conversions_by_channel: pd.Series, H: np.ndarray, channels: list) -> pd.Series:
    vec = conversions_by_channel.values
    haloed = H.dot(vec)
    return pd.Series(haloed, index=channels)

# Example: halo-adjusted conversions for the 'base+500k' scenario
sc = apply_allocation_and_score(panel, greedy_optimize(panel, 500_000, {ch:30 for ch in channels}, objective='roi')[0])
by_ch = sc.groupby('channel')['conversions'].sum().reindex(channels)
halo_adj = apply_halo(by_ch, H, channels)
pd.DataFrame({"raw_conversions": by_ch, "halo_adjusted_conversions": halo_adj})

## 9) Reporting visuals

In [21]:
# Channel-level marginal ROAS from the allocation in Section 5
plt.figure()
plt.bar(summary['channel'], summary['marginal_roas'])
plt.title("Marginal ROAS by Channel")
plt.xlabel("Channel"); plt.ylabel("Marginal ROAS")
plt.show()

# Scenario total ROAS comparison
plt.figure()
plt.bar(totals.index, totals['roas'])
plt.title("Scenario ROAS (Quarter)")
plt.xlabel("Scenario"); plt.ylabel("ROAS")
plt.show()

NameError: name 'summary' is not defined

<Figure size 640x480 with 0 Axes>

## 10) What to deliver to stakeholders

**Outputs you can export:**
1. **Allocation plan**: extra impressions and spend by channel.
2. **Quarterly scenario table**: cost, conversions, revenue, ROAS for each budget level.
3. **Efficiency insights**: CAC and marginal ROAS deltas by channel.
4. **Risk bands**: Monte Carlo ROAS distribution → show expected range.
5. **Cross-channel impact**: Halo-adjusted conversions highlighting spillovers.

**How to use this notebook with your data**
- Replace the CSV path in Section 1 with your file (same column names).
- Tweak `adstock` decay, `hill_saturation` half-saturation, and the halo matrix.
- Change `objective` to `'lift'` if you want to maximize incremental conversions instead of ROI.
- Adjust `freq_caps` and block size in the greedy optimizer to reflect real delivery/frequency constraints.

In [ ]:
nb = new_notebook(cells=cells, metadata={"kernelspec":{"name":"python3","display_name":"Python 3"}})
nbf.write(nb, f"/mnt/data/{nb_name}")

csv_path = f"/mnt/data/{csv_name}"
nb_path = f"/mnt/data/{nb_name}"

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("Sample media inputs (you can download this too)", df.head(20))

(csv_path, nb_path)